# GCP VM Costs (Real Pricing via Cloud Billing API)

Fetches **real** GCP VM pricing including Spot and Committed Use Discounts.

**Data Source:** GCP Cloud Billing Catalog API  
**Output:** `vm_costs` table with cloud = 'GCP'

## Prerequisites
Run `00_Setup_Secrets` notebook first to store GCP service account JSON in secret scope.


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"
TABLE_VM_COSTS = f"{CATALOG}.{SCHEMA}.vm_costs"
TABLE_SKU_REGION_MAPPING = f"{CATALOG}.{SCHEMA}.sku_region_mapping"
SECRET_SCOPE = "lakemeter-credentials"

# Load GCP regions from sku_region_mapping (SOURCE OF TRUTH from dbu_prices)
# This ensures we fetch VM costs for ALL regions that have DBU pricing
regions_df = spark.sql(f"""
    SELECT DISTINCT region_code 
    FROM {TABLE_SKU_REGION_MAPPING} 
    WHERE UPPER(cloud) = 'GCP'
""").toPandas()
GCP_REGIONS = regions_df['region_code'].tolist()

print(f"✅ Config: {CATALOG}.{SCHEMA}")
print(f"📍 Loaded {len(GCP_REGIONS)} GCP regions from sku_region_mapping (dbu_prices source)")
print(f"   Regions: {GCP_REGIONS}")


In [ ]:
%pip install google-cloud-billing -q

import os, json, tempfile, re
from datetime import datetime

# Load GCP credentials from secret scope
gcp_sa_json_raw = dbutils.secrets.get(SECRET_SCOPE, "gcp-service-account-json")

# Parse the JSON first to fix the private_key field properly
# The issue is that \n in private_key might be stored as literal backslash-n
try:
    # Try parsing directly first
    sa_data = json.loads(gcp_sa_json_raw)
except json.JSONDecodeError:
    # If that fails, try fixing common escape issues
    # Replace literal \n (two chars) with actual newline in a way that preserves JSON structure
    fixed_json = gcp_sa_json_raw
    # This handles cases where \\n became \n literally
    fixed_json = re.sub(r'(?<!\\)\\n', '\n', fixed_json)
    try:
        sa_data = json.loads(fixed_json)
    except json.JSONDecodeError:
        # Last resort: fix the private_key field specifically
        sa_data = None
        print("⚠️ Could not parse GCP JSON - will write raw to file")

# Write properly formatted JSON to temp file
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    if sa_data:
        # Write properly formatted JSON
        json.dump(sa_data, f)
        print(f"✅ Service Account: {sa_data.get('client_email', 'unknown')}")
    else:
        # Write raw and hope GCP SDK can handle it
        f.write(gcp_sa_json_raw)
    GCP_SA_JSON_PATH = f.name

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GCP_SA_JSON_PATH

# Get instance types
instance_df = spark.sql(f"SELECT DISTINCT instance_type FROM {TABLE_INSTANCE_RATES} WHERE cloud = 'GCP'").toPandas()
INSTANCE_TYPES = instance_df['instance_type'].tolist()
print(f"📋 Found {len(INSTANCE_TYPES)} GCP instance types")


In [ ]:
# Connect to GCP Cloud Billing API
from google.cloud import billing_v1

try:
    client = billing_v1.CloudCatalogClient()
    services = list(client.list_services())
    compute_service = next((s for s in services if 'Compute Engine' in s.display_name), None)
    
    if compute_service:
        print(f"✅ Connected: {compute_service.display_name}")
        print(f"   Service ID: {compute_service.name}")
    else:
        print("❌ Compute Engine service not found")
except Exception as e:
    print(f"❌ Error: {e}")


In [ ]:
# Fetch Compute Engine SKUs
print("🔄 Fetching GCP Compute Engine SKUs...")
all_skus = list(client.list_skus(parent=compute_service.name))
print(f"✅ Fetched {len(all_skus)} SKUs")


In [ ]:
# Debug: Analyze SKU descriptions and instance type patterns
print("📋 Sample GCP SKU descriptions (CPU/RAM related):")
cpu_ram_skus = [s for s in all_skus if any(k in s.description.lower() for k in ['core', 'cpu', 'ram', 'memory'])]
for i, sku in enumerate(cpu_ram_skus[:30]):
    print(f"   {i+1}. {sku.description[:100]}")

print(f"\n📋 Your instance types from table ({len(INSTANCE_TYPES)} total):")
for inst in sorted(INSTANCE_TYPES)[:25]:
    print(f"   - {inst}")

# Analyze instance type patterns to understand families
from collections import Counter
families_in_table = Counter()
for inst in INSTANCE_TYPES:
    parts = inst.split('-')
    if parts:
        families_in_table[parts[0].lower()] += 1

print(f"\n📊 Instance families in your DBU table:")
for fam, count in families_in_table.most_common():
    print(f"   {fam}: {count} instances")


In [ ]:
# GCP prices are per CPU/RAM, not per machine type
# We need to calculate: price = (vCPUs * CPU_rate) + (RAM_GB * RAM_rate)

# Get machine specs from instance_rates table
instance_specs_df = spark.sql(f"""
    SELECT instance_type, vcpus, memory_gb 
    FROM {TABLE_INSTANCE_RATES} 
    WHERE cloud = 'GCP'
""").toPandas()

MACHINE_SPECS = {}
for _, row in instance_specs_df.iterrows():
    MACHINE_SPECS[row['instance_type']] = {
        'vcpus': float(row['vcpus']),
        'ram_gb': float(row['memory_gb'])
    }

print(f"📋 Machine specs loaded for {len(MACHINE_SPECS)} instance types")

# Normalize instance type names - handle both underscore and dash formats
# e2_highmem_2 -> e2-highmem-2, then family = e2
def normalize_instance_type(inst):
    """Convert underscore format to dash format: e2_highmem_2 -> e2-highmem-2"""
    # Replace underscores with dashes
    normalized = inst.replace('_', '-')
    return normalized

def get_family_from_instance(inst):
    """Extract family from instance type, handling both formats"""
    normalized = normalize_instance_type(inst)
    return normalized.split('-')[0].lower()

# Analyze what families we need
from collections import Counter
needed_families = Counter()
for inst in MACHINE_SPECS.keys():
    fam = get_family_from_instance(inst)
    needed_families[fam] += 1
print(f"\n📊 Families needed from your instance_rates table:")
for fam, count in needed_families.most_common():
    print(f"   {fam}: {count}")

# Extract CPU and RAM rates from SKUs
cpu_rates = {}  # {(family, region, tier): rate}
ram_rates = {}  # {(family, region, tier): rate}

# EXPANDED: All GCP machine families including newer ones and GPU instances
FAMILIES = [
    # General purpose
    'n1', 'n2', 'n2d', 'n4', 'e2', 't2d', 't2a',
    # Compute optimized (including new C4 series)
    'c2', 'c2d', 'c3', 'c3d', 'c4', 'c4a', 'c4d', 'h3',
    # Memory optimized
    'm1', 'm2', 'm3',
    # Accelerator optimized (GPU)
    'a2', 'a3', 'g2',
    # Storage optimized
    'z3',
    # Custom/Others
    'custom'
]

# C4/C4a/C4d are very new - may not be in GCP Billing API yet
# We'll use C3/C3D rates as fallback for these newer families
FAMILY_FALLBACKS = {
    'c4': 'c3',
    'c4a': 'c3',
    'c4d': 'c3d',
    'c2': 'c2d',  # If C2 not found, use C2D
}

# More flexible family matching patterns
import re

def extract_family_from_desc(desc):
    """Extract machine family from SKU description with multiple patterns"""
    desc_lower = desc.lower()
    
    # Try exact patterns first (sorted by length, longest first to avoid partial matches)
    sorted_families = sorted(FAMILIES, key=len, reverse=True)
    for f in sorted_families:
        # Pattern: "N2 Instance Core" or "N2D Instance Ram"
        if re.search(rf'\b{f}\b', desc_lower):
            return f
        # Pattern: "Compute optimized Instance Core running in..." sometimes has C2/C3
        if f'{f} instance' in desc_lower or f'{f} custom' in desc_lower:
            return f
    
    return None

families_found_in_api = set()

for sku in all_skus:
    desc = sku.description
    desc_lower = desc.lower()
    
    # Determine pricing tier
    if 'preemptible' in desc_lower or 'spot' in desc_lower: 
        tier = 'spot'
    elif 'commitment' in desc_lower:
        tier = 'reserved_3y' if '3 year' in desc_lower else 'reserved_1y'
    else: 
        tier = 'on_demand'
    
    # Find machine family with improved matching
    family = extract_family_from_desc(desc)
    
    if family:
        families_found_in_api.add(family)
    else:
        continue
    
    # Check if CPU or RAM
    is_cpu = any(k in desc_lower for k in ['core', 'cpu', 'vcpu'])
    is_ram = any(k in desc_lower for k in ['ram', 'memory'])
    
    if not (is_cpu or is_ram):
        continue
    
    # Extract price
    for pi in sku.pricing_info:
        for tr in pi.pricing_expression.tiered_rates:
            price = tr.unit_price.units + tr.unit_price.nanos / 1e9
            if price > 0:
                for region in sku.service_regions:
                    region_lower = region.lower()
                    if region_lower in GCP_REGIONS:
                        key = (family, region_lower, tier)
                        if is_cpu:
                            cpu_rates[key] = price
                        elif is_ram:
                            ram_rates[key] = price

print(f"\n✅ Found {len(cpu_rates)} CPU rates, {len(ram_rates)} RAM rates")
print(f"\n📊 Families found in GCP API: {sorted(families_found_in_api)}")

# Check coverage
needed_set = set(needed_families.keys())
found_set = families_found_in_api
missing = needed_set - found_set
covered = needed_set & found_set
print(f"\n✅ Families with rates: {sorted(covered)}")
if missing:
    print(f"⚠️ Families MISSING rates: {sorted(missing)}")


In [ ]:
# Calculate prices for each instance type
all_prices = []
PRICING_TIERS = ['on_demand', 'spot', 'reserved_1y', 'reserved_3y']

print("🔄 Calculating prices for each instance type...")

# Track which instances got prices
instances_with_prices = set()
instances_without_prices = set()
fallbacks_used = Counter()

for inst_type, specs in MACHINE_SPECS.items():
    family = get_family_from_instance(inst_type)  # Use the normalized function
    vcpus = specs['vcpus']
    ram_gb = specs['ram_gb']
    
    got_any_price = False
    for region in GCP_REGIONS:
        for tier in PRICING_TIERS:
            key = (family, region, tier)
            cpu_rate = cpu_rates.get(key, 0)
            ram_rate = ram_rates.get(key, 0)
            
            # Try fallback family if primary doesn't have rates
            used_fallback = False
            if cpu_rate == 0 and ram_rate == 0 and family in FAMILY_FALLBACKS:
                fallback_family = FAMILY_FALLBACKS[family]
                fallback_key = (fallback_family, region, tier)
                cpu_rate = cpu_rates.get(fallback_key, 0)
                ram_rate = ram_rates.get(fallback_key, 0)
                if cpu_rate > 0 or ram_rate > 0:
                    used_fallback = True
                    fallbacks_used[f"{family}->{fallback_family}"] += 1
            
            if cpu_rate > 0 or ram_rate > 0:
                total_price = (vcpus * cpu_rate) + (ram_gb * ram_rate)
                source = "GCP Cloud Billing API"
                if used_fallback:
                    source += f" (fallback: {FAMILY_FALLBACKS[family]})"
                all_prices.append({
                    "cloud": "GCP",
                    "region": region,
                    "instance_type": inst_type,
                    "pricing_tier": tier,
                    "payment_option": None,
                    "cost_per_hour": round(total_price, 6),
                    "currency": "USD",
                    "source": source,
                    "fetched_at": datetime.utcnow().isoformat()
                })
                got_any_price = True
    
    if got_any_price:
        instances_with_prices.add(inst_type)
    else:
        instances_without_prices.add(inst_type)

print(f"\n✅ Calculated {len(all_prices)} prices")
print(f"   Instances with prices: {len(instances_with_prices)}/{len(MACHINE_SPECS)}")

for tier, count in sorted(Counter(p['pricing_tier'] for p in all_prices).items()):
    print(f"   - {tier}: {count}")

if fallbacks_used:
    print(f"\n📋 Family fallbacks used:")
    for fb, count in fallbacks_used.most_common():
        print(f"   {fb}: {count} times")

if instances_without_prices:
    print(f"\n⚠️ {len(instances_without_prices)} instances WITHOUT prices:")
    for inst in sorted(instances_without_prices)[:20]:
        fam = get_family_from_instance(inst)
        print(f"   - {inst} (family: {fam})")


In [ ]:
# Preview and save
import pandas as pd
df = pd.DataFrame(all_prices)
df['updated_at'] = datetime.utcnow()

print(f"📊 Preview ({len(df)} rows):")
display(df.head(20))

if len(df) > 0:
    spark_df = spark.createDataFrame(df)
    if spark.catalog.tableExists(TABLE_VM_COSTS):
        spark.sql(f"DELETE FROM {TABLE_VM_COSTS} WHERE cloud = 'GCP'")
        spark_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE_VM_COSTS)
    else:
        spark_df.write.mode("overwrite").saveAsTable(TABLE_VM_COSTS)
    print(f"✅ Saved {len(df)} rows to {TABLE_VM_COSTS}")
else:
    print("⚠️ No data to save")
